<a href="https://colab.research.google.com/github/joshuajhchoi/ai2learn/blob/master/Random_Forest_from_Scratch_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Random Forest classifier from scratch

You can visit the following link to see a version with more explanation.
( https://joshua-mobile-choi-1756.trinket.io/python-3-learning-basic-python-syntax-in-4-hours-for-absolute-beginners#/tasks/extra-work-random-forest-from-scratch)

This Colab Link: https://bit.ly/4hUDx2J
 (Or https://colab.research.google.com/drive/1tASiE6wE-X358y8L0cGzqRmQjOQxnUFd?usp=sharing)

####Phase 1: Importing Libraries and Data Preparation

In [ ]:
import numpy as np
from collections import Counter #For counting the occurrences of elements, used in determining the majority class in a leaf node.
from sklearn.datasets import make_classification # For generating sample data

####Phase 2: Defining the Decision Tree Class

In [ ]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=None):
        self.min_samples_split = min_samples_split # Minimum samples required to split a node
        self.max_depth = max_depth # Maximum depth of the tree
        self.tree = None # Will store the learned tree structure

    def _entropy(self, y):
        # Calculates entropy (impurity) of a set of labels 'y'
        counts = np.bincount(y) # Count occurrences of each class
        probabilities = counts[np.nonzero(counts)] / len(y) # Calculate probabilities
        return -np.sum(probabilities * np.log2(probabilities)) # Entropy formula

    def _information_gain(self, X, y, split_idx):
        # Calculates information gain from a split
        parent_entropy = self._entropy(y) # Entropy before split
        left_indices, right_indices = self._split_data(X, split_idx) # Indices of data points in left/right
        n = len(y)
        n_left, n_right = len(left_indices), len(right_indices)
        if n_left == 0 or n_right == 0: # If a split results in an empty node, gain is 0
            return 0
        weighted_child_entropy = (n_left / n) * self._entropy(y[left_indices]) + \
                                 (n_right / n) * self._entropy(y[right_indices]) # Weighted entropy of children
        return parent_entropy - weighted_child_entropy # Information gain

    def _split_data(self, X, split_idx):
        # Splits the data based on the chosen feature and value
        split_value = X[split_idx[0], split_idx[1]] # Value to split on
        left_indices = np.where(X[:, split_idx[1]] <= split_value)[0] # Indices where feature <= split_value
        right_indices = np.where(X[:, split_idx[1]] > split_value)[0] # Indices where feature > split_value
        return left_indices, right_indices

    def _find_best_split(self, X, y):
        # Finds the best split by maximizing information gain
        best_gain = -1
        best_split = None
        for i in range(X.shape[0]): # Iterate through all data points (potential split values)
            for j in range(X.shape[1]): # Iterate through all features
                split_idx = (i, j)  # (data point index, feature index) defines a potential split
                gain = self._information_gain(X, y, split_idx)
                if gain > best_gain: # Keep track of the best gain and split
                    best_gain = gain
                    best_split = split_idx
        return best_split

    def _build_tree(self, X, y, depth=0):
        # Recursively builds the decision tree
        if len(y) < self.min_samples_split or (self.max_depth is not None and depth >= self.max_depth) or len(set(y)) == 1:
            return Counter(y).most_common(1)[0][0] # Leaf node: return the most frequent class
        best_split = self._find_best_split(X, y) # Find the best split for current node
        if best_split is None: # No split found (e.g., all features are constant)
            return Counter(y).most_common(1)[0][0] # Leaf node
        left_indices, right_indices = self._split_data(X, best_split) # Split data
        left_X, left_y = X[left_indices], y[left_indices]
        right_X, right_y = X[right_indices] , y[right_indices]
        left_subtree = self._build_tree(left_X, left_y, depth + 1) # Build left subtree recursively
        right_subtree = self._build_tree(right_X, right_y, depth + 1) # Build right subtree recursively
        return {'split': best_split, 'left': left_subtree, 'right': right_subtree} # Return the tree structure

    def fit(self, X, y):
        # Fits the decision tree to the data
        self.tree = self._build_tree(X, y)

    def _predict_one(self, x):
        # Predicts the class for a single data point
        subtree = self.tree
        while isinstance(subtree, dict): # Traverse the tree
            split_idx = subtree['split']
            split_value = X[split_idx[0], split_idx[1]]
            if x[split_idx[1]] <= split_value: # Go left or right based on the feature value
                subtree = subtree['left']
            else:
                subtree = subtree['right']
        return subtree # Reached a leaf node, return its class

    def predict(self, X):
        # Predicts the classes for multiple data points
        return np.array([self._predict_one(x) for x in X])

####Phase 3: Defining the Random Forest Class

In [ ]:
class RandomForest:
    def __init__(self, n_estimators=100, min_samples_split=2, max_depth=None, max_features=None):
        self.n_estimators = n_estimators # Number of trees in the forest
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.max_features = max_features # Fraction of features to consider at each split
        self.forest = [] # List to store the trained trees

    def fit(self, X, y):
        # Trains the Random Forest
        n_samples = X.shape[0]
        for _ in range(self.n_estimators):
            # 1. Bagging (Bootstrap Aggregating): Create a random subset of the data (with replacement)
            sample_indices = np.random.choice(n_samples, n_samples, replace=True)
            X_sample, y_sample = X[sample_indices], y[sample_indices]

            # 2. Feature Randomness: Randomly select a subset of features
            if self.max_features is None:
                n_features = X.shape[1]
            else:
                n_features = int(self.max_features * X.shape[1])
            feature_indices = np.random.choice(X.shape[1], n_features, replace=False)
            X_sample = X_sample[:, feature_indices] # Use selected features

            # 3. Train a decision tree on the sampled data and features
            tree = DecisionTree(min_samples_split=self.min_samples_split, max_depth=self.max_depth)
            tree.fit(X_sample, y_sample)
            self.forest.append((tree, feature_indices)) # Store the tree and the used feature indices

    def predict(self, X):
        # Predicts the classes for multiple data points
        predictions = []
        for tree, feature_indices in self.forest: # For each tree
            X_subset = X[:, feature_indices] # Use the same features as during training
            predictions.append(tree.predict(X_subset)) # Get predictions from the individual tree
        # Ensemble: Majority voting
        final_predictions = np.array([Counter(preds).most_common(1)[0][0] for preds in zip(*predictions)])
        return final_predictions

####Phase 4: Data Generation

In [ ]:
# Generate some sample data (replace with your data)
X, y = make_classification(n_samples=100, n_features=5, n_informative=3, n_redundant=1, random_state=42)

####Phase 5: Model Training

In [ ]:
#Create and train the Random Forest model
rf_model = RandomForest(n_estimators=50, max_depth=4)
rf_model.fit(X, y)

####Phase 6: Prediction

In [ ]:
predictions = rf_model.predict(X)

####Phase 7: Prediction Result

In [ ]:
print(predictions)

[0 1 1 1 1 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 1 0 1 0 1 1 0 1 1 1 1 1 1 1 0 1 1
 1 1 1 0 0 1 0 1 1 1 1 0 0 0 0 1 0 0 0 1 1 1 1 0 1 1 0 1 1 1 1 1 1 0 0 1 1
 0 0 0 1 0 0 0 0 1 0 0 1 1 1 1 0 0 1 0 0 0 0 1 0 1 0]


####Phase 8: Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y, predictions)
print(f"Accuracy: {accuracy}")

Accuracy: 0.93
